In [36]:
# CREATE TABLE fact_result (
#     result_id serial primary key,
#     session_id int not null,
#     driver_id int not null,
#     team_id int not null,
#     grid_position int,
#     finish_position int,
#     classified_position varchar(5),
#     points numeric,
#     status varchar(50),
#     q1_time_s numeric,
#     q2_time_s numeric,
#     q3_time_s numeric,
#     race_time_s numeric,         -- czas zwycięzcy / różnica do lidera

#     CONSTRAINT fk_result_session FOREIGN KEY (session_id) REFERENCES dim_session(session_id),
#     CONSTRAINT fk_result_driver  FOREIGN KEY (driver_id)  REFERENCES dim_driver(driver_id),
#     CONSTRAINT fk_result_team    FOREIGN KEY (team_id)    REFERENCES dim_team(team_id)
# );
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import fastf1

fastf1.Cache.enable_cache('/tmp/fastf1_cache')

spark = SparkSession.builder \
        .appName("F1_Telemetry_Processing") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2") \
        .getOrCreate()

url = "jdbc:postgresql://localhost:5432/telemetry"
properties = {"user": "michal", "password": "root", "driver": "org.postgresql.Driver"}

In [11]:
dim_season = spark.read.jdbc(url=url, table='dim_season', properties=properties)
dim_season_df = dim_season.collect()
dim_session = spark.read.jdbc(url=url, table='dim_session', properties=properties)
dim_session_df = dim_session.collect()
dim_driver = spark.read.jdbc(url=url, table='dim_driver', properties=properties)
dim_driver_df = dim_driver.collect()
dim_team = spark.read.jdbc(url=url, table='dim_team', properties=properties)
dim_team_df = dim_team.collect()
dim_event = spark.read.jdbc(url=url, table='dim_event', properties=properties)
dim_event_df = dim_event.collect()

season_map = {row['season_id']: row['year'] for row in dim_season_df}
driver_map = {row['code']: row['driver_id'] for row in dim_driver_df}
team_map = {row['name']: row['team_id'] for row in dim_team_df}
event_map = {
    row["event_id"]: (row["season_id"], row["round_number"])
    for row in dim_event.collect()
}

In [72]:
def build_fact_result(session, session_id, driver_map, team_map, spark):        
    results = session.results
    time_cols = ["Q1", "Q2", "Q3", "Time"]
    for col in time_cols:
        if col in results.columns:
            results[col] = results[col].dt.total_seconds()

    result_df = spark.createDataFrame(results)
    result_df = result_df.withColumn("session_id", F.lit(session_id))

    driver_mapping = F.create_map([F.lit(x) for pair in driver_map.items() for x in pair])
    team_mapping   = F.create_map([F.lit(x) for pair in team_map.items()   for x in pair])
    
    result_df = result_df.withColumn("driver_id", driver_mapping[F.col("Abbreviation")])
    result_df = result_df.withColumn("team_id",   team_mapping[F.col("TeamName")])
    
    result_df = result_df.select(["session_id",
                                  "driver_id",
                                  "team_id",
                                  "GridPosition",
                                  "Position",
                                  "ClassifiedPosition",
                                  "Points",
                                  "Status",
                                  "Q1",
                                  "Q2",
                                  "Q3",
                                  "Time"])
    result_df = result_df.withColumnsRenamed({"GridPosition": "grid_position",
                                               "Position": "finish_position",
                                               "ClassifiedPosition": "classified_position",
                                               "Points": "points",
                                               "Status": "status",
                                               "Q1": "q1_time_s",
                                               "Q2": "q2_time_s",
                                               "Q3": "q3_time_s",
                                               "Time": "race_time_s"})
    
    float_cols = [f.name for f in result_df.schema.fields
                  if isinstance(f.dataType, (DoubleType, FloatType))]
    for col in float_cols:
        result_df = result_df.withColumn(col,
            F.when(F.isnan(F.col(col)), None).otherwise(F.col(col)))

    str_cols = [f.name for f in result_df.schema.fields
                if str(f.dataType) == "StringType()"]
    for col in str_cols:
        result_df = result_df.withColumn(col,
            F.when(F.col(col) == "", None).otherwise(F.col(col)))
    
    result_df = result_df \
        .withColumn("grid_position",   F.col("grid_position").cast(IntegerType()))
    return result_df

In [73]:
loaded = 0
errors = 0
for row in dim_session_df:
    season_id, round_number = event_map[row.event_id]
    if row.session_type not in ["R", "Q", "S", "SQ"]:
        continue
    try:
        session = fastf1.get_session(season_map[season_id], round_number, row.session_type)
        session.load()
    except:
        print(f"Faild to load for {(season_map[season_id], round_number, row.session_type)}")
        errors += 1
        continue
    sdf = build_fact_result(session, row.session_id, driver_map, team_map, spark)
    sdf.write.jdbc(url=url, table='fact_result', mode='append', properties=properties)
    loaded += 1
print(f'loaded {loaded/(loaded+errors)*100}% ({loaded}, {errors})')

core           INFO 	Loading data for Austrian Grand Prix - Sprint Qualifying [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Austrian Grand Prix - Sprint Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
core        WARNING 	Sprint Qualifying is not supported by Ergast! Limited results are calculated from timing data.
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached da

Faild to load for (2025, 8, 'Q')


core           INFO 	Loading data for Azerbaijan Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Azerbaijan Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


Faild to load for (2019, 4, 'Q')


core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Monaco Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


...

Faild to load for (2024, 8, 'Q')


core           INFO 	Loading data for Canadian Grand Prix - Qualifying [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Canadian Grand Prix - Qualifying [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


Faild to load for (2024, 8, 'R')


core           INFO 	Loading data for Canadian Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Canadian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


Faild to load for (2024, 9, 'Q')
Faild to load for (2024, 9, 'R')
Faild to load for (2022, 15, 'Q')
Faild to load for (2022, 15, 'R')
Faild to load for (2022, 16, 'Q')
Faild to load for (2022, 16, 'R')
